In [1]:

#@title 1. NEW Mount google drive & install dependencies
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
#@title NEW 1.1 Install dependencies
# !pip install -q transformers datasets accelerate evaluate jiwer soundfile librosa torchaudio pandas#
!pip install -q transformers datasets accelerate evaluate jiwer soundfile librosa torchaudio pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 64.6 MB/s eta 0:00:00


In [ ]:
#@title NEW 1.2 When you mess up the unzipping
# !rm -rf /content/speecht5-malagasy-asr/
# !rmdir /content/speecht5-malagasy-asr/

In [3]:
#@title NEW 1.3 Pre config additionnelles
'''
EDIT NEW USAGE: Extract NEXT 1000 Badrex examples (skip first 2000)
'''

# CHEMIN_LATEST_CHECKPOINT_ZIP = "/content/drive/MyDrive/Projet M2/speecht5-malagasy-asr-20260911_162823.zip"
# CHEMIN_LATEST_CHECKPOINT = "/content/speecht5-malagasy-asr/final"
CHEMIN_LATEST_CHECKPOINT = "/content/drive/MyDrive/speecht5-malagasy-asr/final"

# !mkdir -p "{CHEMIN_LATEST_CHECKPOINT}"
# !unzip -q "{CHEMIN_LATEST_CHECKPOINT_ZIP}" -d /content/

# print(f"Latest checkpoint locally extracted to: {CHEMIN_LATEST_CHECKPOINT}")
print(f"Latest checkpoint locally extracted from: {CHEMIN_LATEST_CHECKPOINT}")

Latest checkpoint locally extracted from: /content/drive/MyDrive/speecht5-malagasy-asr/final


In [4]:
#@title 2. NEW Imports & Configuration
import os
import re
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from datasets import Dataset, Audio, DatasetDict
from transformers import (
    SpeechT5Processor,
    SpeechT5ForSpeechToText,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import evaluate

# ---------- Configuration ----------
# ROOT_DIR = "/content/dataset_badrex"   # ← never used in this new version v1.1 using badrex
OUTPUT_DIR= "/content/drive/MyDrive/speecht5-malagasy-asr"    # ← persistent location on Drive
# OUTPUT_DIR = "/content/speecht5-malagasy-asr"
CHECKPOINT = CHEMIN_LATEST_CHECKPOINT

#@title 9. Resume training from checkpoint-800
# CHECKPOINT = os.path.join(OUTPUT_DIR, "checkpoint-800")

print("Configuration loaded.")
# print(f"ROOT_DIR   : {ROOT_DIR}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")

Configuration loaded.
OUTPUT_DIR : /content/drive/MyDrive/speecht5-malagasy-asr


In [5]:
#@title 3. NEW Extract limited Badrex dataset (~2000 examples)
'''
EDIT NEW USAGE: Extract NEXT 1000 Badrex examples (skip first 2000)
'''
from datasets import load_dataset
import soundfile as sf
from tqdm import tqdm
import time
import shutil # for training next 1000 rows of dataset

# NB_EXEMPLES = 2000
SKIP = 2000 # nbr of old rows already trained on
NB_EXEMPLES = 1000
DOSSIER_SORTIE = "/content/dataset_badrex"          # ← single source of truth

shutil.rmtree(DOSSIER_SORTIE, ignore_errors=True) # just in case there are leftover dataset
os.makedirs(DOSSIER_SORTIE, exist_ok=True)

# print(f"Extracting {NB_EXEMPLES} examples from badrex/malagasy-speech-full...\n")
print(f"Skipping first {SKIP}, then extracting {NB_EXEMPLES}...") # NEW SKIPPING USAGE

dataset = load_dataset(
    "badrex/malagasy-speech-full",
    split="train",
    streaming=True
)

compteur = 0
duree_totale = 0.0
debut = time.time()
vus = 0 # FOR SKIPPING ITERATIONS

# for exemple in tqdm(dataset, desc="Extraction"):
for exemple in tqdm(dataset, desc="Skip + extract"): # SKIPPING USAGE
    # SKIP
    vus +=1
    if vus <= SKIP:
        continue

    if compteur >= NB_EXEMPLES:
        break

    audio = exemple.get("audio")
    transcription = exemple.get("transcription")

    if not audio or not transcription:
        continue

    nom = f"badrex_{compteur:05d}"
    wav_path = f"{DOSSIER_SORTIE}/{nom}.wav"
    txt_path = f"{DOSSIER_SORTIE}/{nom}.txt"

    sf.write(wav_path, audio["array"], 16000)

    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(transcription.strip())

    duree_totale += exemple.get("audio_duration", 0)
    compteur += 1

taille = sum(
    os.path.getsize(os.path.join(DOSSIER_SORTIE, f))
    for f in os.listdir(DOSSIER_SORTIE)
) / (1024 ** 3)

print(f"\n{'='*60}")
print(f"✅ EXTRACTION TERMINÉE")
print(f"{'='*60}")
print(f"Exemples     : {compteur}")
print(f"Durée audio  : {duree_totale/3600:.2f} heures")
print(f"Temps        : {(time.time()-debut)/60:.1f} min")
print(f"Taille       : {taille:.2f} Go")
print(f"Dossier      : {DOSSIER_SORTIE}")

Skipping first 2000, then extracting 1000...


README.md:   0%|          | 0.00/834 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/23 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/23 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/207 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/23 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/23 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/207 [00:00<?, ?it/s]

Skip + extract: 3000it [03:50, 13.01it/s]



✅ EXTRACTION TERMINÉE
Exemples     : 1000
Durée audio  : 5.25 heures
Temps        : 3.8 min
Taille       : 0.56 Go
Dossier      : /content/dataset_badrex


**Q:** what does the new 6th cell do ? And does that make the 4th cell useless ?

**A:** Yes. With the new Cell 6, Cell 4 is useless. Skip it.
What the new Cell 6 does
It walks the extracted files in /content/dataset_badrex (1.wav + 1.txt, etc.) and builds a lightweight Hugging Face dataset.
For each pair it:

Reads the .txt transcription
Checks duration with soundfile.info (does not load the waveform into RAM)
Tokenizes the text with truncation=True, max_length=450
Drops junk (too short/long audio, empty text)
Stores only:
audio_path — path to the wav on disk
labels — token ids of the text

Splits into train / test (8% eval)

It does not store input_values (the actual audio). That was what blew RAM.
Audio is loaded later, one batch at a time, in the new Cell 7 collator.

In [ ]:
#@title 4. USELESS NEW Build Dataset from extracted Badrex files
from datasets import Dataset, Audio

pairs = []
for wav_file in sorted(Path(DOSSIER_SORTIE).glob("*.wav")):
    txt_file = wav_file.with_suffix(".txt")
    if not txt_file.exists():
        continue

    with open(txt_file, "r", encoding="utf-8") as f:
        text = f.read().strip()

    if text:
        pairs.append({
            "audio": str(wav_file),
            "text": text
        })

print(f"Valid pairs found: {len(pairs)}")

raw_dataset = Dataset.from_list(pairs)
raw_dataset = raw_dataset.cast_column("audio", Audio(sampling_rate=16000))

dataset = raw_dataset.train_test_split(test_size=0.08, seed=42)

print(dataset)
print(f"\nTrain size : {len(dataset['train'])}")
print(f"Eval  size : {len(dataset['test'])}")

Valid pairs found: 2000
DatasetDict({
    train: Dataset({
        features: ['audio', 'text'],
        num_rows: 1840
    })
    test: Dataset({
        features: ['audio', 'text'],
        num_rows: 160
    })
})

Train size : 1840
Eval  size : 160


In [ ]:
#@title 4.1 USELESS Inspect right after Cell 4
print("Dataset structure right after Cell 4:")
print(dataset)
print()
print(f"Train size: {len(dataset['train'])}")
print(f"Test size : {len(dataset['test'])}")
print()

if len(dataset["train"]) > 0:
    example = dataset["train"][0]
    print("First example keys:", list(example.keys()))
    print("Text sample:", example.get("text", "")[:100])
    print("Audio type :", type(example.get("audio")))
else:
    print("⚠ Dataset is empty even after Cell 4!")

Dataset structure right after Cell 4:
DatasetDict({
    train: Dataset({
        features: ['audio', 'text'],
        num_rows: 1840
    })
    test: Dataset({
        features: ['audio', 'text'],
        num_rows: 160
    })
})

Train size: 1840
Test size : 160

First example keys: ['audio', 'text']
Text sample: Fanafody izay vita amin'ny kininim-potsy. Ankafizin'ny olona tokoa itony fanafody itony ary vita mal
Audio type : <class 'datasets.features._torchcodec.AudioDecoder'>


In [7]:


#@title 5. SAME Load SpeechT5 processor & model
'''
EDIT NEW USAGE: Train on NEXT 1000 Badrex examples (skip first 2000)
'''
processor = SpeechT5Processor.from_pretrained(CHECKPOINT)
# processor = SpeechT5Processor.from_pretrained('/content/speecht5-malagasy-asr/final')
model = SpeechT5ForSpeechToText.from_pretrained(CHECKPOINT)
model.config.use_cache = False          # required for gradient checkpointing

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"Model loaded on: {device}")
print(f"Number of parameters: {model.num_parameters():,}")

Loading weights:   0%|          | 0/369 [00:00<?, ?it/s]

Model loaded on: cuda
Number of parameters: 151,165,056


In [ ]:
#@title 6. NEW DOESN'T WORK Preprocess function (handles torchcodec + length filtering)
import numpy as np

def prepare_dataset(batch):
    try:
        audio = batch["audio"]

        # ----- Handle different audio formats -----
        if audio is None:
            return None

        # New format (torchcodec AudioDecoder)
        if hasattr(audio, "get_all_samples"):
            samples = audio.get_all_samples()
            array = samples.data.numpy().squeeze()
            sr = samples.sample_rate
        # Old format (dict)
        elif isinstance(audio, dict) and "array" in audio:
            array = audio["array"]
            sr = audio.get("sampling_rate", 16000)
        else:
            return None

        # Convert to mono if needed
        if array.ndim > 1:
            array = array.mean(axis=0)

        # Resample if necessary
        if sr != 16000:
            import librosa
            array = librosa.resample(array, orig_sr=sr, target_sr=16000)

        duration = len(array) / 16000

        # Filter too short / too long audio
        if duration < 0.5 or duration > 30.0:
            return None

        # Process speech
        batch["input_values"] = processor(
            audio=array,
            sampling_rate=16000
        ).input_values[0]

        # Process text
        with processor.as_target_processor():
            labels = processor(text=batch["text"]).input_ids

            if len(labels) < 2 or len(labels) > 400:
                return None

            batch["labels"] = labels

        return batch

    except Exception as e:
        # Uncomment the next line if you want to see the errors
        # print(f"Skipping sample: {e}")
        return None


print("Preprocessing + filtering...")

dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset["train"].column_names,
    num_proc=1,
    load_from_cache_file=False,
    desc="Preprocessing"
)

# Remove failed samples
dataset = dataset.filter(lambda x: x is not None and "input_values" in x)

print("✅ Preprocessing done!")
print(dataset)
print(f"Train size after filtering : {len(dataset['train'])}")
print(f"Eval  size after filtering : {len(dataset['test'])}")

Preprocessing + filtering...


Preprocessing (num_proc=1):   0%|          | 0/1840 [00:00<?, ? examples/s]

Preprocessing (num_proc=1):   0%|          | 0/160 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1840 [00:00<?, ? examples/s]

Filter:   0%|          | 0/160 [00:00<?, ? examples/s]

✅ Preprocessing done!
DatasetDict({
    train: Dataset({
        features: ['audio', 'text'],
        num_rows: 0
    })
    test: Dataset({
        features: ['audio', 'text'],
        num_rows: 0
    })
})
Train size after filtering : 0
Eval  size after filtering : 0


In [ ]:
#@title 6. DEBUG version – find out why everything is filtered
import numpy as np
from collections import Counter

reasons = Counter()

def prepare_dataset_debug(batch):
    try:
        audio = batch["audio"]
        text = batch.get("text", "")

        # 1. Check audio existence
        if audio is None:
            reasons["audio is None"] += 1
            return None

        # 2. Extract waveform
        if hasattr(audio, "get_all_samples"):          # torchcodec
            samples = audio.get_all_samples()
            array = samples.data.numpy().squeeze()
            sr = samples.sample_rate
        elif isinstance(audio, dict) and "array" in audio:  # old style
            array = np.array(audio["array"])
            sr = audio.get("sampling_rate", 16000)
        else:
            reasons[f"unknown audio type: {type(audio)}"] += 1
            return None

        if array.ndim > 1:
            array = array.mean(axis=0)

        if sr != 16000:
            import librosa
            array = librosa.resample(y=array.astype(np.float32), orig_sr=sr, target_sr=16000)

        duration = len(array) / 16000

        if duration < 0.5:
            reasons["too short audio"] += 1
            return None
        if duration > 30.0:
            reasons["too long audio"] += 1
            return None

        # 3. Process speech
        processed = processor(audio=array, sampling_rate=16000)
        input_values = processed.input_values[0]

        # 4. Process text
        with processor.as_target_processor():
            labels = processor(text=text).input_ids

        if len(labels) < 2:
            reasons["text too short"] += 1
            return None
        if len(labels) > 400:
            reasons["text too long"] += 1
            return None

        # Success
        batch["input_values"] = input_values
        batch["labels"] = labels
        return batch

    except Exception as e:
        reasons[f"EXCEPTION: {type(e).__name__}: {str(e)[:80]}"] += 1
        return None


print("Running diagnostic preprocessing on a few samples...")

# Test on a small subset first
small = dataset["train"].select(range(min(50, len(dataset["train"]))))
_ = small.map(prepare_dataset_debug, num_proc=1, load_from_cache_file=False)

print("\n===== REASONS WHY SAMPLES WERE SKIPPED =====")
for reason, count in reasons.most_common():
    print(f"{count:4d} × {reason}")

Running diagnostic preprocessing on a few samples...

===== REASONS WHY SAMPLES WERE SKIPPED =====


In [ ]:
#@title 6b. Direct inspection of the first few samples
import numpy as np

print("Inspecting the first 5 samples of the training set...\n")

for i in range(min(5, len(dataset["train"]))):
    example = dataset["train"][i]
    print(f"===== Sample {i} =====")
    print(f"Keys          : {list(example.keys())}")
    print(f"Text (first 80): {str(example.get('text', ''))[:80]}")

    audio = example.get("audio")
    print(f"Audio type    : {type(audio)}")

    if audio is None:
        print("Audio is None")
    elif hasattr(audio, "get_all_samples"):
        print("Detected torchcodec AudioDecoder")
        try:
            samples = audio.get_all_samples()
            print(f"  sample_rate : {samples.sample_rate}")
            print(f"  data shape  : {samples.data.shape}")
        except Exception as e:
            print(f"  Error calling get_all_samples: {e}")
    elif isinstance(audio, dict):
        print("Detected dict-style audio")
        print(f"  keys        : {list(audio.keys())}")
        if "array" in audio:
            arr = np.array(audio["array"])
            print(f"  array shape : {arr.shape}")
            print(f"  sampling_rate: {audio.get('sampling_rate')}")
    else:
        print(f"Unknown audio format: {audio}")

    print()

Inspecting the first 5 samples of the training set...



In [ ]:
#@title 6. Preprocess (never return None, respect 450 token limit)
import numpy as np

# Silence "460 > 450" and avoid indexing errors
processor.tokenizer.model_max_length = 450

def prepare_dataset(batch):
    audio = batch["audio"]
    text = batch["text"]

    samples = audio.get_all_samples()
    array = samples.data
    sr = int(samples.sample_rate)

    if array.ndim > 1:
        array = array.mean(dim=0)
    array = array.cpu().numpy().astype(np.float32)

    if sr != 16000:
        import librosa
        array = librosa.resample(array, orig_sr=sr, target_sr=16000)

    duration = len(array) / 16000

    # Always produce labels (truncate so it never exceeds 450)
    labels = processor.tokenizer(
        text,
        truncation=True,
        max_length=450,   # must be ≤ 450, this is the model limit
        add_special_tokens=True,
    ).input_ids

    speech = processor(audio=array, sampling_rate=16000)

    batch["input_values"] = speech.input_values[0]
    batch["labels"] = labels
    batch["duration"] = duration
    batch["label_length"] = len(labels)
    batch["was_truncated"] = len(text) > 0 and labels[-1] != processor.tokenizer.eos_token_id
    return batch   # ALWAYS return a dict


print("Preprocessing dataset...")

dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset["train"].column_names,
    num_proc=1,
    load_from_cache_file=False,
    desc="Preprocessing",
)

# Filter AFTER map (this is the safe way to drop samples)
dataset = dataset.filter(
    lambda x: (x["duration"] >= 0.5)
              and (x["duration"] <= 30.0)
              and (2 <= x["label_length"] <= 400)
)

print("✅ Preprocessing done!")
print(dataset)
print(f"Train size : {len(dataset['train'])}")
print(f"Eval  size : {len(dataset['test'])}")

Preprocessing dataset...


Preprocessing (num_proc=1):   0%|          | 0/1840 [00:00<?, ? examples/s]

RuntimeError: One of the subprocesses has abruptly died during map operation.To debug the error, disable multiprocessing.

In [ ]:
#@title 6. NEW ATTEMPT NO 42069 Preprocess in the MAIN process (no map, no multiprocessing)
import numpy as np
from datasets import Dataset, DatasetDict

processor.tokenizer.model_max_length = 450

def process_split(split, name="split"):
    rows = []
    skipped = 0

    for i, ex in enumerate(split):
        try:
            text = ex["text"]
            audio = ex["audio"]

            samples = audio.get_all_samples()
            array = samples.data
            sr = int(samples.sample_rate)

            if array.ndim > 1:
                array = array.mean(dim=0)
            array = array.cpu().numpy().astype(np.float32)

            if sr != 16000:
                import librosa
                array = librosa.resample(array, orig_sr=sr, target_sr=16000)

            duration = len(array) / 16000.0
            if duration < 0.5 or duration > 30.0:
                skipped += 1
                continue

            # truncation HERE so 460 > 450 never happens
            labels = processor.tokenizer(
                text,
                truncation=True,
                max_length=450,
                add_special_tokens=True,
            ).input_ids

            if len(labels) < 2:
                skipped += 1
                continue

            input_values = processor(
                audio=array,
                sampling_rate=16000,
            ).input_values[0]

            rows.append({
                "input_values": np.asarray(input_values, dtype=np.float32),
                "labels": labels,
            })

        except Exception as e:
            skipped += 1
            if skipped <= 15:
                print(f"[{name} #{i}] {type(e).__name__}: {e}")
            continue

        if (i + 1) % 100 == 0:
            print(f"{name}: {i+1}/{len(split)}  kept={len(rows)}  skipped={skipped}")

    print(f"{name} done: kept={len(rows)}  skipped={skipped}")
    return Dataset.from_list(rows)


print("Re-run Cell 4 first if dataset is empty, then this cell.")
print(f"Input sizes: train={len(dataset['train'])}  test={len(dataset['test'])}")

dataset = DatasetDict({
    "train": process_split(dataset["train"], "train"),
    "test":  process_split(dataset["test"],  "test"),
})

print("✅ Preprocessing done!")
print(dataset)

Re-run Cell 4 first if dataset is empty, then this cell.
Input sizes: train=1840  test=160
train: 100/1840  kept=99  skipped=1
train: 200/1840  kept=199  skipped=1
train: 300/1840  kept=299  skipped=1
train: 400/1840  kept=399  skipped=1
train: 500/1840  kept=499  skipped=1
train: 600/1840  kept=599  skipped=1
train: 700/1840  kept=698  skipped=2
train: 800/1840  kept=796  skipped=4
train: 900/1840  kept=894  skipped=6
train: 1000/1840  kept=994  skipped=6
train: 1100/1840  kept=1094  skipped=6
train: 1200/1840  kept=1194  skipped=6
train: 1300/1840  kept=1294  skipped=6
train: 1400/1840  kept=1394  skipped=6
train: 1500/1840  kept=1493  skipped=7
train: 1600/1840  kept=1593  skipped=7
train: 1700/1840  kept=1693  skipped=7
train: 1800/1840  kept=1792  skipped=8
train done: kept=1832  skipped=8


In [8]:
#@title 6. NEW ATTEMPMT No 69420 RAM-safe preprocess (paths + labels only, no waveforms in RAM)
'''
EDIT NEW USAGE: Extract NEXT 1000 Badrex examples (skip first 2000)
'''
import numpy as np
from pathlib import Path
from datasets import Dataset, DatasetDict
import soundfile as sf

DOSSIER_SORTIE = "/content/dataset_badrex"
processor.tokenizer.model_max_length = 450

rows = []
skipped = 0

wav_files = sorted(Path(DOSSIER_SORTIE).glob("*.wav"))
print(f"Found {len(wav_files)} wav files")

for i, wav_file in enumerate(wav_files):
    txt_file = wav_file.with_suffix(".txt")
    if not txt_file.exists():
        skipped += 1
        continue

    text = txt_file.read_text(encoding="utf-8").strip()
    if not text:
        skipped += 1
        continue

    try:
        duration = sf.info(str(wav_file)).duration
    except Exception:
        skipped += 1
        continue

    if duration < 0.5 or duration > 30.0:
        skipped += 1
        continue

    labels = processor.tokenizer(
        text,
        truncation=True,
        max_length=450,
        add_special_tokens=True,
    ).input_ids

    if len(labels) < 2:
        skipped += 1
        continue

    rows.append({
        "audio_path": str(wav_file),
        "labels": labels,
    })

    if (i + 1) % 200 == 0:
        print(f"{i+1}/{len(wav_files)}  kept={len(rows)}  skipped={skipped}")

print(f"Done: kept={len(rows)}  skipped={skipped}")

# for SKIPPING
from datasets import DatasetDict

raw = Dataset.from_list(rows)
# SKIP
# dataset = raw.train_test_split(test_size=0.08, seed=42)
dataset = DatasetDict({"train": raw}) # NEW USAGE: TO TRAIN ON THE NEXT 1000  DATASET; 100% train, no test

print(dataset)
# print(f"Train: {len(dataset['train'])} | Test: {len(dataset['test'])}")
print(f"Train: {len(dataset['train'])} | Test: 0")


Found 1000 wav files
200/1000  kept=198  skipped=2
400/1000  kept=397  skipped=3
600/1000  kept=595  skipped=5
800/1000  kept=795  skipped=5
1000/1000  kept=995  skipped=5
Done: kept=995  skipped=5
DatasetDict({
    train: Dataset({
        features: ['audio_path', 'labels'],
        num_rows: 995
    })
})
Train: 995 | Test: 0


In [9]:
#@title 7. NEW data collator (loads wav per batch) & WER metric
import librosa
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch
import evaluate

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], str]]]) -> Dict[str, torch.Tensor]:
        waves = []
        for f in features:
            array, _ = librosa.load(f["audio_path"], sr=16000, mono=True)
            waves.append(array)

        batch = self.processor.feature_extractor(
            waves,
            sampling_rate=16000,
            padding=True,
            return_tensors="pt",
        )

        labels_batch = self.processor.tokenizer.pad(
            {"input_ids": [f["labels"] for f in features]},
            return_tensors="pt",
        )
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1), -100
        )
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": round(wer, 2)}

print("Collator and metrics ready.")

Collator and metrics ready.


In [11]:
#@title 8. NEW Training Arguments (Colab-safe, saves to Drive every 100 steps)
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,                       # saves directly to Google Drive
    per_device_train_batch_size=1,               # On-the-fly wav loading + long clips OOM (out of memory) easily
    per_device_eval_batch_size=1,                # also changed 2 -> 1, no explanation, flemme demander
    gradient_accumulation_steps=8,               # effective batch size = 16
    learning_rate=5e-5,

    # warmup_steps=100,
    warmup_steps=50, # NEW USAGE: TO TRAIN ON THE NEXT 1000  DATASET

    # num_train_epochs=5,                          # 1830 examples × 20 epochs ≈ 2300 steps — Colab will die first → 5 epochs ; 20 was for 315 data available
    num_train_epochs=3, # NEW USAGE: TO TRAIN ON THE NEXT 1000  DATASET
    gradient_checkpointing=True,
    fp16=True,

    # --- Colab resilience settings ---

    # --- NEW USAGE: TO TRAIN ON THE NEXT 1000  DATASET ---
        # delete: eval_steps, metric_for_best_model, greater_is_better
        # predict_with_generate can stay or go; it only matters for eval

    # eval_strategy="steps",
    eval_strategy="no",                           # NEW USAGE: TO TRAIN ON THE NEXT 1000  DATASET
    # eval_steps=200,                              # predict_with_generate=True on 160 clips every 100 steps is slow and RAM-heavy
    save_strategy="steps",
    save_steps=200,                              # save checkpoint every 200 steps to Drive; Must match eval when load_best_model_at_end=True
    save_total_limit=2,                          # keep only the last 2 checkpoints
    # load_best_model_at_end=True,
    load_best_model_at_end=False,                # NEW USAGE: TO TRAIN ON THE NEXT 1000  DATASET
    # metric_for_best_model="wer",
    # greater_is_better=False,

    # predict_with_generate=True,
    generation_max_length=120,
    logging_steps=50,
    report_to=["tensorboard"],
    push_to_hub=False,
    remove_unused_columns=False
)

print("Training arguments set.")
print(f"Checkpoints will be saved every {training_args.save_steps} steps to:")
print(OUTPUT_DIR)

# Then Cell 9.
# If GPU OOM still happens, set predict_with_generate=False (eval will use loss only, no WER until the end).

Training arguments set.
Checkpoints will be saved every 200 steps to:
/content/drive/MyDrive/speecht5-malagasy-asr


**NOTE:** The Trainer is dropping `audio_path` before the collator runs.
By default `remove_unused_columns=True` . The model’s `forward()` does not take `audio_path`, so Hugging Face deletes it. Then the collator does `f["audio_path"]` and crashes.
Add this to Cell 8 and re-run Cell 8, then Cell 9 (no need to re-run 6 or 7):
```
remove_unused_columns=False,
```

In [12]:
#@title 9. NEW Create Trainer & Start Training

'''
EDIT NEW USAGE: Extract NEXT 1000 Badrex examples (skip first 2000)
'''

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],

    # eval_dataset=dataset["test"],     # NEW USAGE: TO TRAIN ON THE NEXT 1000  DATASET; removed
    data_collator=data_collator,
    # tokenizer=processor.feature_extractor, # was commented in OLD version because bad argument, let's see with this new vers

    # compute_metrics=compute_metrics,  # NEW USAGE: TO TRAIN ON THE NEXT 1000  DATASET; removed (optional; unused without eval)
)

# print("🚀 Starting training...")
# trainer.train()
print("Resuming from", CHECKPOINT)
# trainer.train(resume_from_checkpoint=True) # to train from checkpoints when training was interrupted instead of completed
'''
Do not use resume_from_checkpoint=True. That job already finished (1145/1145). Resuming it will often do 0 new steps.
Load the weights from your last checkpoint, then start a new run on the new 1000 clips.
'''
trainer.train()

Resuming from /content/drive/MyDrive/speecht5-malagasy-asr/final


Step,Training Loss
50,2.477139
100,2.751221
150,2.511701
200,2.241549
250,2.212109
300,2.013570
350,1.987970


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=375, training_loss=2.2871358133951825, metrics={'train_runtime': 1308.597, 'train_samples_per_second': 2.281, 'train_steps_per_second': 0.287, 'total_flos': 8.154483546204864e+17, 'train_loss': 2.2871358133951825, 'epoch': 3.0})

In [13]:
#@title 10. NEW Save final model (to Google Drive)
final_path = os.path.join(OUTPUT_DIR, "final2")

trainer.save_model(final_path)
processor.save_pretrained(final_path)

print(f"✅ Final model saved to: {final_path}")
!ls -la "{final_path}"

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Final model saved to: /content/drive/MyDrive/speecht5-malagasy-asr/final2
total 590791
drwx------ 2 root root      4096 Sep 16 22:17 .
drwx------ 6 root root      4096 Sep 16 22:17 ..
-rw------- 1 root root        40 Sep 16 22:17 added_tokens.json
-rw------- 1 root root      2061 Sep 16 22:17 config.json
-rw------- 1 root root       920 Sep 16 22:17 generation_config.json
-rw------- 1 root root 604711192 Sep 16 22:17 model.safetensors
-rw------- 1 root root       492 Sep 16 22:17 processor_config.json
-rw------- 1 root root    238473 Sep 16 22:17 spm_char.model
-rw------- 1 root root      1399 Sep 16 22:17 tokenizer_config.json
-rw------- 1 root root      5393 Sep 16 22:17 training_args.bin


In [14]:
#@title 11. NEW Inference function for any MP3 file
import librosa

def transcribe_mp3(mp3_path, model, processor, max_length=120, device="cuda"):
    speech, sr = librosa.load(mp3_path, sr=16000, mono=True)

    inputs = processor(audio=speech, sampling_rate=16000, return_tensors="pt")
    input_values = inputs.input_values.to(device)

    with torch.no_grad():
        predicted_ids = model.generate(input_values, max_length=max_length)

    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return transcription


# Example usage
mp3_file = "/content/dataset_badrex/badrex_00100.wav"   # change this path

prediction = transcribe_mp3(
    mp3_path=mp3_file,
    model=model,
    processor=processor,
    max_length=450,
    device=device
)

print("Predicted transcription:")
print(prediction)

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
[transformers] Caching is incompatible with gradient checkpointing in SpeechT5DecoderLayer. Setting `past_key_values=None`.


Predicted transcription:
a  a f a   n  - n  a  a ,  f  ,  a h      a f a   f a  , a  f  a - a a nf   a     a     a a a -   a           a a f  a    a    a    a -  a  a a   a a a a a   a  a   a     a      a   f  a n a a   h  n f a a    a a    a - a     a    a , a a ,  a     - a       a o  a   f  a  a -     a - a f a  f a     t a -  -   a  a a  a   -      a  a     a   f  ,   a a    .  a a f  a a  a  n    a  a a  a -    a - ,   v  a  a       a o a  a      f a  a   a    o f


In [17]:
#@title 11. NEW BETTER Inference (eval mode, no grad checkpoint)
'''
LET'S SEE IF THE INFERENCE FUNCTION IS WRONG OR IF THE TRAINING AND THE NEW WEIGHTS WERE TRASH
→ The inference function was wrong, unadapted to the pure train/ no eval model

'''
import librosa
import torch

def transcribe_mp3(mp3_path, model, processor, max_length=450, device="cuda"):
    model.eval()
    model.gradient_checkpointing_disable()
    model.config.use_cache = True

    speech, sr = librosa.load(mp3_path, sr=16000, mono=True)
    inputs = processor(audio=speech, sampling_rate=16000, return_tensors="pt")
    # input_values = inputs.input_values.to(device)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        # predicted_ids = model.generate(input_values, max_length=max_length)
        predicted_ids = model.generate(
            **inputs,
            max_length=max_length,
            min_length=2,
            num_beams=4,
        )


    # print("token ids (first 40):", predicted_ids[0, :40].tolist())
    return processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]


mp3_file = "/content/dataset_badrex/badrex_00100.wav"
print("REF:", open(mp3_file.replace(".wav", ".txt"), encoding="utf-8").read())
print("HYP:", transcribe_mp3(mp3_file, model, processor, device=device))

REF: Fitaovana vaovao elektronika iray ampiasaina amin'ny fampahenana ny hafanana ao anaty efitrano. Manana endrika mahitsizoro mitsangana ary vita amin'ny plastika fotsy ; misy latabatra fanaraha-maso izay hafahan'ny mpampiasa manitsy ny maripana sy ny hafainganam-pandehan'ny rivotra. Misy kojakoja maro hafa toy ny sivana rivotra sy ny fantsona fivoahan'ny rivotra mafana.
HYP: Fitaovana vaovao elektoranika iray ampiasaina mifampahinana ny hafananao anaty efitrano. Manana endrika mahitsy zoro mitsangana ary vita ampilastika fotsy. Misy latabatra fanaraha maso izay hafahany mpampiasa manitsy ny marompana sy ny hafaingana mpandehany rivotra misy koja koja maro afatra ny sivana rivotra sy ny fantsona fivoan'ny rivotra mafana.


In [ ]:
# Example usage
mp3_file = "/content/micro_kindindrenty.mp3"   # change this path

prediction = transcribe_mp3(
    mp3_path=mp3_file,
    model=model,
    processor=processor,
    max_length=450,
    device=device
)

print("Predicted transcription:")
print(prediction)

Predicted transcription:
Kindrindray an'ity.


In [ ]:
# Example usage
mp3_file = "/content/audio.mp3"   # change this path

prediction = transcribe_mp3(
    mp3_path=mp3_file,
    model=model,
    processor=processor,
    max_length=450,
    device=device
)

print("Predicted transcription:")
print(prediction)

Predicted transcription:
Tsara ny andro.


In [ ]:
# Example usage
mp3_file = "/content/Recteur.input.wav"   # change this path

prediction = transcribe_mp3(
    mp3_path=mp3_file,
    model=model,
    processor=processor,
    max_length=450,
    device=device
)

print("Predicted transcription:")
print(prediction)

Predicted transcription:
Ny ankizindry dia manaikin'iray mandreny sy mianara tsara any am-pianarany.


In [18]:
# Example usage
# mp3_file = "/content/dataset_badrex/badrex_00400.wav"   # change this path
mp3_file = "/content/drive/MyDrive/Projet M2/echantillons_dataset/cleaned/0031/1.mp3"   # change this path

prediction = transcribe_mp3(
    mp3_path=mp3_file,
    model=model,
    processor=processor,
    max_length=450,
    device=device
)

print("Predicted transcription:")
print(prediction)

Predicted transcription:
Bia ny sisy tokofaraka amin'ny telopolo nasaina andriamanitra ny verina ny betelahy jakoba. Ary jakoba na ireny teny ny zanakalain'ny labana na na hoe efa lasa ny jakoba avokoa izay rehetra nanana ny raintsika.




---


# OLD FROM V1.0



---



In [ ]:
#@title 1. OLD Mount Google Drive & Install dependencies
from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers datasets accelerate evaluate jiwer soundfile librosa torchaudio pandas

# ------------------------------------------------------------------
# IMPORTANT:
# After mounting, you must set the correct path to your "cleaned" folder
# inside your Google Drive.
#
# Example:
#   If the folder is at:  My Drive / cleaned
#   then the path is:     /content/drive/MyDrive/cleaned
#
#   If it is inside another folder, adjust accordingly.
# ------------------------------------------------------------------

ROOT_DIR = "/content/drive/MyDrive/Projet M2/echantillons_dataset/cleaned"   # ← change this if needed

print("Google Drive mounted.")
print(f"Looking for data in: {ROOT_DIR}")

# Quick check
import os
from pathlib import Path

if Path(ROOT_DIR).exists():
    print("\n✅ Folder found! Contents:")
    !ls -la "{ROOT_DIR}" | head -20
else:
    print("\n❌ Folder not found. Please check the path and update ROOT_DIR.")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 797.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 9.3 MB/s eta 0:00:00
Google Drive mounted.
Looking for data in: /content/drive/MyDrive/Projet M2/echantillons_dataset/cleaned

✅ Folder found! Contents:
ls: '/content/drive/MyDrive/Projet M2/echantillons_dataset/cleaned/0031': No such file or directory
ls: '/content/drive/MyDrive/Projet M2/echantillons_dataset/cleaned/0032': No such file or directory
ls: '/content/drive/MyDrive/Projet M2/echantillons_dataset/cleaned/0042': No such file or directory
ls: '/content/drive/MyDrive/Projet M2/echantillons_dataset/cleaned/0033': No such file or directory
ls: '/content/drive/MyDrive/Projet M2/echantillons_dataset/cleaned/0034': No such file or directory
ls: '/content/drive/MyDrive/Projet M2/echantillons_dataset/cleaned/0035': No such file or directory
ls: '/content/drive/MyDrive/Projet M2/echantillons_dataset/cleaned/0036': No such file

In [ ]:
#@title 2. OLD Imports & Configuration
import os
import re
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from datasets import Dataset, Audio, DatasetDict
from transformers import (
    SpeechT5Processor,
    SpeechT5ForSpeechToText,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import evaluate

# ---------- Configuration ----------
# ROOT_DIR = "/content/cleaned"          # where the numbered folders are
OUTPUT_DIR = "./speecht5-malagasy-asr"
CHECKPOINT = "microsoft/speecht5_asr"

print("Configuration loaded.")
print(f"ROOT_DIR  : {ROOT_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

Configuration loaded.
ROOT_DIR  : /content/drive/MyDrive/Projet M2/echantillons_dataset/cleaned
OUTPUT_DIR: ./speecht5-malagasy-asr


In [ ]:
#@title 3. OLD Collect all (audio, text) pairs from every subfolder
def load_all_pairs(root_dir: str):
    pairs = []
    root = Path(root_dir)

    for subfolder in sorted(root.iterdir()):
        if not subfolder.is_dir():
            continue

        csv_path = subfolder / "phrase.csv"
        if not csv_path.exists():
            print(f"⚠ Skipping {subfolder.name}: no phrase.csv")
            continue

        with open(csv_path, "r", encoding="utf-8") as f: # , errors="ignore"
            for line in f:
                line = line.strip()
                if not line or "|" not in line:
                    continue

                # Split only on the first |
                number, text = line.split("|", 1)
                # Clean the 'number' to remove any hidden characters like BOM
                number = number.strip().lstrip('\ufeff') # Added .lstrip('\ufeff')
                text = text.strip()

                audio_path = subfolder / f"{number}.mp3"
                if audio_path.exists() and text:
                    pairs.append({
                        "audio": str(audio_path),
                        "text": text,
                        "folder": subfolder.name,
                        "id": f"{subfolder.name}_{number}"
                    })
                else:
                    print(f"  Missing: {audio_path.name}")

    print(f"\n✅ Total valid pairs found: {len(pairs)}")
    return pairs

pairs = load_all_pairs(ROOT_DIR)

print("\n--- First 5 examples ---")
for p in pairs[:5]:
    print(f"{p['id']}: {p['text'][:70]}...")


✅ Total valid pairs found: 350

--- First 5 examples ---
0031_1: Nasain'Andriamanintra niverina any betela i Jakoba Ary Jakoba nandre n...
0031_2: ary avy tamin'ny fananan'ny raintsika no nahazoany izao voninahitra re...
0031_3: ary homba anao Aho. Ary Jakoba naniraka ka nampaka an-dRahely sy Lea h...
0031_4: Ary fantatrareo fa tamin'ny heriko rehetra no nanompoako ny rainareo. ...
0031_5: dia niteraka mara ny ondry aman'osy rehetra: ary raha nilaza hoe indra...


In [ ]:
#@title 4. OLD Build Dataset and split train/eval
raw_dataset = Dataset.from_list(pairs)
raw_dataset = raw_dataset.cast_column("audio", Audio(sampling_rate=16000))

# 90/10 split (good for small data)
dataset = raw_dataset.train_test_split(test_size=0.1, seed=42)

print(dataset)
print(f"\nTrain size: {len(dataset['train'])}")
print(f"Eval  size: {len(dataset['test'])}")

DatasetDict({
    train: Dataset({
        features: ['audio', 'text', 'folder', 'id'],
        num_rows: 315
    })
    test: Dataset({
        features: ['audio', 'text', 'folder', 'id'],
        num_rows: 35
    })
})

Train size: 315
Eval  size: 35


In [ ]:
#@title 5. SAME OLD Load SpeechT5 processor & model
processor = SpeechT5Processor.from_pretrained(CHECKPOINT)
model = SpeechT5ForSpeechToText.from_pretrained(CHECKPOINT)
model.config.use_cache = False          # required for gradient checkpointing

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"Model loaded on: {device}")
print(f"Number of parameters: {model.num_parameters():,}")

In [ ]:
#@title 6. OLD Preprocess function (audio → input_values, text → labels)
def prepare_dataset(batch):
    audio = batch["audio"]

    # Extract features
    batch["input_values"] = processor(
        audio=audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_values[0]

    # Tokenize Malagasy text
    # Changed from 'with processor.as_target_processor():'
    batch["labels"] = processor.tokenizer(text=batch["text"]).input_ids

    return batch

print("Preprocessing dataset (this may take a few minutes)...")
dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset["train"].column_names,
    num_proc=1,          # safer with mp3
)

print("✅ Preprocessing done!")
print(dataset)


In [ ]:
#@title 7. SAME OLD Data collator & WER metric
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # pad speech features
        input_features = [{"input_values": f["input_values"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # pad labels
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # mask padding tokens in labels
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# Word Error Rate
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": round(wer, 2)}

print("Data collator and metrics ready.")

In [ ]:
#@title 8. OLD Training Arguments (small-data friendly)
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,       # effective batch = 16
    learning_rate=5e-5,
    warmup_steps=100,
    max_steps=400,                      # number of epochs ~= max_steps / 20; increase if you have more data later (initially 1500 ~= 75 epochs (3 hours of waiting time))
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps", # Changed from evaluation_strategy="steps"
    eval_steps=200,
    save_steps=200,
    logging_steps=50,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=120,
    save_total_limit=3,
    push_to_hub=False,
)

print("Training arguments set.")

Training arguments set.


In [ ]:
#@title 9. OLD Create Trainer & Start Training
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    # tokenizer=processor.tokenizer, # Removed this line
    compute_metrics=compute_metrics,
)

print("🚀 Starting training...")
trainer.train()

🚀 Starting training...


Step,Training Loss,Validation Loss,Wer
200,3.541443,0.389153,72.280000
400,2.161055,0.328969,67.590000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['text_decoder_postnet.lm_head.weight'].


TrainOutput(global_step=400, training_loss=6.033257751464844, metrics={'train_runtime': 2756.6048, 'train_samples_per_second': 2.322, 'train_steps_per_second': 0.145, 'total_flos': 1.5447371524402084e+18, 'train_loss': 6.033257751464844, 'epoch': 20.0})

In [ ]:
#@title 10. OLD Save final model
final_path = os.path.join(OUTPUT_DIR, "final")

trainer.save_model(final_path)
processor.save_pretrained(final_path)

print(f"✅ Model saved to: {final_path}")
!ls -la {final_path}

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to: ./speecht5-malagasy-asr/final
total 590816
drwxr-xr-x 2 root root      4096 Sep 11 16:19 .
drwxr-xr-x 6 root root      4096 Sep 11 16:19 ..
-rw-r--r-- 1 root root        40 Sep 11 16:19 added_tokens.json
-rw-r--r-- 1 root root      2061 Sep 11 16:19 config.json
-rw-r--r-- 1 root root       920 Sep 11 16:19 generation_config.json
-rw------- 1 root root 604711192 Sep 11 16:19 model.safetensors
-rw-r--r-- 1 root root       492 Sep 11 16:19 processor_config.json
-rw-r--r-- 1 root root    238473 Sep 11 16:19 spm_char.model
-rw-r--r-- 1 root root      1400 Sep 11 16:19 tokenizer_config.json
-rw-r--r-- 1 root root      5393 Sep 11 16:19 training_args.bin


In [ ]:
#@title 11. OLD Quick inference test on one sample
from random import randint

# Pick one example from the test set
number = randint(0, len(dataset["test"]))
example = dataset["test"][number]
input_values = torch.tensor(example["input_values"]).unsqueeze(0).to(device)

with torch.no_grad():
    predicted_ids = model.generate(input_values, max_length=120)

transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
print("Predicted :", transcription)

# If you want to hear the original audio you can reload it, but for simplicity we just print the text

Predicted : ary maina ny lavaka tamin'izay fa tsy nisy rano tao, Ary nipetraka ny anakaniny izy ireo ary sendra nanopy ny masony k


In [ ]:
#@title 11.2 OLD Inference function + test on one example
def transcribe(example, model, processor, max_length=120, device="cuda"):
    """
    Transcribe a single preprocessed example.

    Args:
        example: a sample from the dataset (must contain "input_values")
        model: the fine-tuned SpeechT5 model
        processor: the SpeechT5 processor
        max_length: maximum length of the generated transcription
        device: "cuda" or "cpu"
    """
    input_values = torch.tensor(example["input_values"]).unsqueeze(0).to(device)

    with torch.no_grad():
        predicted_ids = model.generate(input_values, max_length=max_length)

    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return transcription


# Test on the first example of the evaluation set
example = dataset["test"][0]
prediction = transcribe(example, model, processor, max_length=120, device=device)

print("Predicted transcription:")
print(prediction)

Predicted transcription:
Ary Andriamanita dia nijery ny zanak'i Jiraely ka nahafantatra azy,


In [ ]:
#@title 11.3 OLD Inference function for any MP3 file
import torchaudio
import librosa
import soundfile as sf
from pathlib import Path

def transcribe_mp3(mp3_path, model, processor, max_length=120, device="cuda"):
    """
    Transcribe any MP3 file using the fine-tuned SpeechT5 model.

    Args:
        mp3_path (str): path to the .mp3 file
        model: the fine-tuned SpeechT5 model
        processor: the SpeechT5 processor
        max_length (int): maximum length of the generated transcription
        device (str): "cuda" or "cpu"

    Returns:
        str: the predicted transcription
    """
    # Load audio and force 16 kHz mono
    speech, sr = librosa.load(mp3_path, sr=16000, mono=True)

    # Process with the SpeechT5 processor
    inputs = processor(audio=speech, sampling_rate=16000, return_tensors="pt")
    input_values = inputs.input_values.to(device)

    # Generate transcription
    with torch.no_grad():
        predicted_ids = model.generate(input_values, max_length=max_length)

    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return transcription


# -------------------------------------------------
# Example usage
# -------------------------------------------------
# mp3_file = "/content/drive/MyDrive/cleaned/0061/1.mp3" # ← change this path
mp3_file = f"{ROOT_DIR}/0061/1.mp3"

prediction = transcribe_mp3(
    mp3_path=mp3_file,
    model=model,
    processor=processor,
    max_length=120,
    device=device
)

print("Predicted transcription:")
print(prediction)

Predicted transcription:
Aksodosy toko faraika ambin'ny folo,


In [ ]:
#@title 11.4 OLD test with arbitrary audio 1
prediction = transcribe_mp3(
    mp3_path="/content/audio.mp3",
    model=model,
    processor=processor,
    max_length=120,
    device=device
)

print("Predicted transcription:")
print(prediction)

Predicted transcription:
Asahara nihiandro,


In [ ]:
#@title 11.5 OLD test with arbitrary audio 2
prediction = transcribe_mp3(
    mp3_path="/content/micro_kindindrenty.mp3",
    model=model,
    processor=processor,
    max_length=120,
    device=device
)

print("Predicted transcription:")
print(prediction)

Predicted transcription:
sy indian'i Jenity,




---



# NOTES



---



**Q:**
 See the folder "speecht5-malagasy-asr" ? This is what popped up when I executed the 10th cell (Save final model). Now that I want to use this model in my app, what exactly in that folder should I download ? Or do I download the entire folder ?

**A:** En ce qui concerne les fichiers à télécharger de votre dossier speecht5-malagasy-asr/final pour votre application, vous devez télécharger l'intégralité du dossier final. Ce dossier contient tous les éléments nécessaires pour utiliser votre modèle fine-tuné et son processeur. Voici les fichiers importants et leur rôle:

  - `model.safetensors` ou `pytorch_model.bin` : C'est le fichier qui contient les poids entraînés de votre modèle SpeechT5 fine-tuné. C'est le cœur de votre modèle.

  - `config.json` : Contient la configuration de l'architecture de votre modèle.

  - `generation_config.json` : Contient la configuration pour la génération de texte (par exemple, la longueur maximale de la séquence, la stratégie de décodage).

  - `processor_config.json`, `tokenizer_config.json`, `spm_char.model` et `added_tokens.json` : Ces fichiers sont tous nécessaires pour le SpeechT5Processor. Ils définissent comment l'audio est traité et comment le texte est tokenisé et détokenisé. Sans eux, le modèle ne peut pas préparer correctement les entrées audio ou décoder les sorties textuelles.

  - `training_args.bin` : Contient les arguments d'entraînement utilisés, utile pour la traçabilité mais pas strictement nécessaire pour l'inférence.

En téléchargeant le dossier `final` complet, vous vous assurez d'avoir tous les composants pour charger et utiliser votre modèle fine-tuné dans votre application.


### Charger le modèle et le processeur fine-tunés

**Q:** Now that I have downloaded the final folder, how do I do to load it up ?

**A:** Pour charger le modèle et le processeur à partir de votre dossier `final`, vous utilisez la même méthode `from_pretrained` que précédemment, mais en spécifiant le chemin de votre dossier local au lieu d'un nom de point de contrôle Hugging Face Hub.

In [ ]:
from transformers import SpeechT5Processor, SpeechT5ForSpeechToText
import torch

# Le chemin vers le dossier 'final' que vous avez sauvegardé
# Assurez-vous que ce chemin est correct si vous déplacez le dossier
loaded_model_path = "./speecht5-malagasy-asr/final"

# Charger le processeur
loaded_processor = SpeechT5Processor.from_pretrained(loaded_model_path)

# Charger le modèle
loaded_model = SpeechT5ForSpeechToText.from_pretrained(loaded_model_path)

# Déplacer le modèle vers le GPU si disponible
device = "cuda" if torch.cuda.is_available() else "cpu"
loaded_model = loaded_model.to(device)

print(f"Modèle et processeur chargés depuis : {loaded_model_path}")
print(f"Modèle chargé sur : {device}")

### Sauvegarder les points de contrôle et reprendre l'entraînement

**Q:**
And in the eventuality that I will add more data to train from , how can I make checkpoints so that when the Colab tokens run out my work aren't gone ?

**A:** Vos `Seq2SeqTrainingArguments` (définis dans la Cellule 8) incluent déjà les paramètres nécessaires à la sauvegarde des points de contrôle :

*   `save_steps=200` : Cela indique au *Trainer* de sauvegarder un point de contrôle toutes les 200 étapes d'entraînement.
*   `save_total_limit=3` : Cela signifie que seuls les 3 derniers points de contrôle seront conservés (les plus anciens seront supprimés). Pour conserver tous les points de contrôle, vous pouvez définir ceci sur `None` ou un nombre très élevé.

Ces points de contrôle sont sauvegardés dans votre `output_dir` (qui est `./speecht5-malagasy-asr`). Vous trouverez des dossiers nommés `checkpoint-XXX` (par exemple, `checkpoint-200`, `checkpoint-400`, etc.) à l'intérieur de ce répertoire.

**Pour reprendre l'entraînement** :

Lorsque votre session Colab expire ou se déconnecte, vous pouvez simplement relancer les cellules précédentes (jusqu'à la Cellule 9 incluse) pour recharger vos données et votre configuration. Ensuite, vous pouvez dire au *Trainer* de reprendre l'entraînement à partir du dernier point de contrôle sauvegardé en utilisant le paramètre `resume_from_checkpoint`.

In [ ]:
# Relancer le training à partir du dernier checkpoint
print("🚀 Reprise de l'entraînement...")
# Le trainer trouvera automatiquement le dernier checkpoint dans OUTPUT_DIR
trainer.train(resume_from_checkpoint=True)

# Alternativement, vous pouvez spécifier un chemin de checkpoint exact si vous le souhaitez:
# last_checkpoint = "./speecht5-malagasy-asr/checkpoint-400" # Remplacez par le chemin réel
# trainer.train(resume_from_checkpoint=last_checkpoint)

### Liste des dossiers deja trained from
0031
0032
0033
0034
0035
0036
0037
0038
0039
0040
0041
0042
0051
0052
0061

### Télécharger le dossier `speecht5-malagasy-asr`

Vous pouvez créer un fichier zip du dossier, puis le télécharger directement depuis Colab.

In [ ]:
# Format for file names (No spaces or colons): YYYYMMDD_HHMMSS
from datetime import datetime

now = datetime.now()

filename_ts = now.strftime("%Y%m%d_%H%M%S")

# Compresser le dossier
!zip -r "/content/speecht5-malagasy-asr-{filename_ts}.zip" ./speecht5-malagasy-asr/final

# Télécharger le fichier zip
# from google.colab import files
# files.download('/content/speecht5-malagasy-asr.zip')

print("✅ Le dossier 'speecht5-malagasy-asr' a été compressé et le téléchargement est lancé.")

  adding: speecht5-malagasy-asr/final/ (stored 0%)
  adding: speecht5-malagasy-asr/final/processor_config.json (deflated 49%)
  adding: speecht5-malagasy-asr/final/training_args.bin (deflated 54%)
  adding: speecht5-malagasy-asr/final/generation_config.json (deflated 59%)
  adding: speecht5-malagasy-asr/final/added_tokens.json (deflated 13%)
  adding: speecht5-malagasy-asr/final/spm_char.model (deflated 42%)
  adding: speecht5-malagasy-asr/final/config.json (deflated 68%)
  adding: speecht5-malagasy-asr/final/tokenizer_config.json (deflated 78%)
  adding: speecht5-malagasy-asr/final/model.safetensors (deflated 7%)
✅ Le dossier 'speecht5-malagasy-asr' a été compressé et le téléchargement est lancé.
